# Pitch Vision — Stage 4 (Formation / Network Graph)

Builds on Stage 3 (tracking, team colors, possession, camera motion, homography, speed/distance) by adding a flat, to-scale top-down "formation graph": each team's players as colored dots on a schematic pitch, connected to their nearest teammates by lines — the classic broadcast tactical-overlay look, generated automatically from the real-world (meters) positions Stage 3 already computed. This is Module A milestone A1 from the roadmap.

Rendered as its OWN separate video (not drawn on top of the drone footage) — watch it side by side with the main tracking video.

This clip's camera is a static full-pitch overhead shot, so camera motion should come out near zero — the optical-flow step is there so the pipeline still gives correct results if a future clip ever comes from a camera that pans or drifts.

**One manual step is required and can't be automated away safely:** the homography needs to know where 4 known pitch points (e.g. the four corners) are in PIXEL coordinates. A later cell displays your clip's first frame with a coordinate grid so you can read those off by eye — much more reliable than guessing. It's pre-filled below with the corners you already read off for this clip in Stage 3.

**Before running:** Runtime → Change runtime type → GPU (T4 is enough).

In [ ]:
!pip install -q ultralytics supervision
import os
for d in ['utils', 'trackers', 'team_assigner', 'player_ball_assigner',
          'camera_movement_estimator', 'view_transformer', 'speed_and_distance_estimator',
          'formation_graph']:
    os.makedirs(d, exist_ok=True)
print("Project folders created.")

In [ ]:
from google.colab import files
print("Upload best_topview.pt (your Stage 1 top-view fine-tuned model):")
uploaded = files.upload()
model_path = list(uploaded.keys())[0]
print("Using model:", model_path)

In [ ]:
print("Upload clip_01_0-12s.mp4 (from your project's input_videos/ folder):")
uploaded_clip = files.upload()
clip_path = list(uploaded_clip.keys())[0]
print("Using clip:", clip_path)

## Writing the pipeline modules

In [ ]:
%%writefile utils/__init__.py


In [ ]:
%%writefile trackers/__init__.py


In [ ]:
%%writefile team_assigner/__init__.py


In [ ]:
%%writefile player_ball_assigner/__init__.py


In [ ]:
%%writefile camera_movement_estimator/__init__.py


In [ ]:
%%writefile view_transformer/__init__.py


In [ ]:
%%writefile speed_and_distance_estimator/__init__.py


In [ ]:
%%writefile utils/bbox_utils.py
def get_center_of_bbox(bbox):
    x1, y1, x2, y2 = bbox
    return int((x1 + x2) / 2), int((y1 + y2) / 2)


def get_bbox_width(bbox):
    return bbox[2] - bbox[0]


def get_foot_position(bbox):
    x1, y1, x2, y2 = bbox
    return int((x1 + x2) / 2), int(y2)


def measure_distance(p1, p2):
    return ((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2) ** 0.5


In [ ]:
%%writefile utils/video_utils.py
import cv2


def read_video(video_path, target_width=1920):
    # Resize down while reading, not after — our clip is native 4K (3840x2160), and
    # holding all ~360 frames in memory at full 4K is ~9GB on its own, enough to crash
    # Colab's free-tier RAM by itself. target_width=1920 matches the resolution our
    # training data was stored at, so detection stays consistent with training too.
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        h, w = frame.shape[:2]
        if target_width is not None and w > target_width:
            scale = target_width / w
            frame = cv2.resize(frame, (target_width, int(h * scale)), interpolation=cv2.INTER_AREA)
        frames.append(frame)
    cap.release()
    return frames


def get_native_frame(video_path, frame_idx):
    """
    Reads exactly ONE frame directly from the source video file at its original,
    un-downsampled resolution (e.g. native 4K even though read_video() gives back
    1920-wide frames for detection/tracking/drawing).

    Used only for team-color sampling: a player crop that's already tiny gets made
    even blurrier by the resize read_video() does to keep RAM usage sane, which
    contaminates shirt-color sampling with blended-in grass pixels. Re-reading just
    the handful of frames we actually need color from, at full detail, avoids that
    without holding the whole clip in memory at 4K.
    """
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f"Could not read frame {frame_idx} from {video_path}")
    return frame


def save_video(output_video_frames, output_video_path, fps=25):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    h, w = output_video_frames[0].shape[:2]
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))
    for frame in output_video_frames:
        out.write(frame)
    out.release()


In [ ]:
%%writefile trackers/tracker.py
import os
import pickle
import sys

import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO

sys.path.append('/content')
from utils.bbox_utils import get_center_of_bbox, get_bbox_width


class Tracker:
    def __init__(self, model_path):
        self.model = YOLO(model_path)
        self.tracker = sv.ByteTrack()

    def detect_frames(self, frames, conf=0.2, imgsz=1280, batch_size=20):
        # imgsz=1280 matches training — inference at the default 640 would shrink the
        # ball back down below what the model was actually trained to recognize.
        detections = []
        for i in range(0, len(frames), batch_size):
            batch = self.model.predict(frames[i:i + batch_size], conf=conf, imgsz=imgsz, verbose=False)
            detections += batch
        return detections

    def get_object_tracks(self, frames, read_from_stub=False, stub_path=None):
        if read_from_stub and stub_path is not None and os.path.exists(stub_path):
            with open(stub_path, 'rb') as f:
                return pickle.load(f)

        detections = self.detect_frames(frames)

        tracks = {"players": [], "ball": []}

        for frame_num, detection in enumerate(detections):
            cls_names = detection.names  # {0: 'player', 1: 'ball'}
            cls_names_inv = {v: k for k, v in cls_names.items()}

            detection_supervision = sv.Detections.from_ultralytics(detection)
            detection_with_tracks = self.tracker.update_with_detections(detection_supervision)

            tracks["players"].append({})
            tracks["ball"].append({})

            # players get persistent track_ids from ByteTrack
            for frame_detection in detection_with_tracks:
                bbox = frame_detection[0].tolist()
                cls_id = frame_detection[3]
                track_id = frame_detection[4]
                if cls_id == cls_names_inv.get('player'):
                    tracks["players"][frame_num][track_id] = {"bbox": bbox}

            # the ball gets a hardcoded track_id of 1 — there's only ever one, no need to
            # track its identity across frames, just its position
            for frame_detection in detection_supervision:
                bbox = frame_detection[0].tolist()
                cls_id = frame_detection[3]
                if cls_id == cls_names_inv.get('ball'):
                    tracks["ball"][frame_num][1] = {"bbox": bbox}

        if stub_path is not None:
            with open(stub_path, 'wb') as f:
                pickle.dump(tracks, f)

        return tracks

    def draw_ellipse(self, frame, bbox, color, track_id=None):
        y2 = int(bbox[3])
        x_center, _ = get_center_of_bbox(bbox)
        width = get_bbox_width(bbox)

        cv2.ellipse(
            frame,
            center=(x_center, y2),
            axes=(int(width), int(0.35 * width)),
            angle=0.0,
            startAngle=-45,
            endAngle=235,
            color=color,
            thickness=2,
            lineType=cv2.LINE_4,
        )

        rect_w, rect_h = 40, 20
        x1_rect = x_center - rect_w // 2
        x2_rect = x_center + rect_w // 2
        y1_rect = (y2 - rect_h // 2) + 15
        y2_rect = (y2 + rect_h // 2) + 15

        if track_id is not None:
            cv2.rectangle(frame, (int(x1_rect), int(y1_rect)), (int(x2_rect), int(y2_rect)), color, cv2.FILLED)
            x1_text = x1_rect + 12
            if track_id > 99:
                x1_text -= 10
            cv2.putText(frame, f"{track_id}", (int(x1_text), int(y1_rect + 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

        return frame

    def draw_triangle(self, frame, bbox, color):
        y = int(bbox[1])
        x, _ = get_center_of_bbox(bbox)
        points = np.array([[x, y], [x - 10, y - 20], [x + 10, y - 20]])
        cv2.drawContours(frame, [points], 0, color, cv2.FILLED)
        cv2.drawContours(frame, [points], 0, (0, 0, 0), 2)
        return frame

    def draw_annotations(self, video_frames, tracks, team_ball_control):
        output_frames = []
        for frame_num, frame in enumerate(video_frames):
            frame = frame.copy()
            player_dict = tracks["players"][frame_num]
            ball_dict = tracks["ball"][frame_num]

            for track_id, player in player_dict.items():
                color = player.get("team_color", (0, 0, 255))
                frame = self.draw_ellipse(frame, player["bbox"], color, track_id)
                if player.get("has_ball", False):
                    frame = self.draw_triangle(frame, player["bbox"], (0, 0, 255))

            for _, ball in ball_dict.items():
                frame = self.draw_triangle(frame, ball["bbox"], (0, 255, 0))

            if len(team_ball_control) > 0 and frame_num < len(team_ball_control):
                so_far = team_ball_control[:frame_num + 1]
                t1 = int((so_far == 1).sum())
                t2 = int((so_far == 2).sum())
                total = t1 + t2
                if total > 0:
                    cv2.putText(frame, f"Team 1 Ball Control: {t1 / total * 100:.1f}%", (50, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
                    cv2.putText(frame, f"Team 2 Ball Control: {t2 / total * 100:.1f}%", (50, 90),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

            output_frames.append(frame)
        return output_frames


In [ ]:
%%writefile team_assigner/team_assigner.py
import cv2
import numpy as np
from sklearn.cluster import KMeans


class TeamAssigner:
    """
    Assigns each tracked person to a team by clustering shirt color, and — since our
    detector only has a 'player' class (no separate 'referee' class in training data) —
    flags anyone whose shirt color doesn't cleanly match either team cluster as a
    non-player (referee) instead of forcing them into the nearest team.

    History of what didn't work, kept here because the next person touching this file
    (possibly future-me) will otherwise re-try the same dead ends:
      1. Top-half-of-box sampling (broadcast/side-view assumption: shirt on top, shorts
         below) -- wrong for an overhead view, where the top of a tiny box is head/hair.
      2. Full-box + inner 2-cluster KMeans, treating corner pixels as "background" --
         fails on tight boxes where the corners are still the player's own body.
      3. Plain median of the whole box, even the whole native-resolution box -- still
         failed in practice. Measured real output looked like BGR (91, 133, 109) and
         (101, 145, 124) for the two teams: nearly identical AND both green-dominant
         (G channel highest in both) -- a dead giveaway that grass pixels, not shirt
         pixels, were winning the median vote. A generously-sized/loosely-fit detection
         box around a small player can be majority background even when "tight" by eye.
    """

    # Grass in this footage (real turf, mowing stripes) sits in a fairly consistent
    # green hue band regardless of light/dark stripe -- stripes differ in brightness
    # (V), not hue. OpenCV hue is 0-179. Saturation gate avoids excluding dark/desaturated
    # shirts that merely happen to fall in the same hue range.
    GRASS_HUE_LOW = 25
    GRASS_HUE_HIGH = 95
    GRASS_SAT_MIN = 40

    def __init__(self):
        self.team_colors = {}
        self.player_team_dict = {}
        self.kmeans = None
        self.outlier_threshold = None

    def get_player_color(self, frame, bbox):
        x1, y1, x2, y2 = [int(v) for v in bbox]
        image = frame[y1:y2, x1:x2]
        if image.size == 0:
            return np.array([0, 0, 0])

        # Trim a small margin off each edge -- the outermost pixels of even a tight box
        # are the most likely to be anti-aliased/motion-blurred blends with whatever's
        # just outside the player.
        h, w = image.shape[:2]
        my, mx = int(h * 0.1), int(w * 0.1)
        if h - 2 * my > 0 and w - 2 * mx > 0:
            image = image[my:h - my, mx:w - mx]

        # Explicitly drop grass-hued pixels before summarizing color. This is a more
        # direct fix than just hoping a tighter crop avoids background: it targets the
        # actual contamination (green pitch) by color, so it still works even when the
        # detection box itself is loose or the player is tiny and blurry.
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        hue, sat = hsv[:, :, 0], hsv[:, :, 1]
        is_grass = (hue >= self.GRASS_HUE_LOW) & (hue <= self.GRASS_HUE_HIGH) & (sat >= self.GRASS_SAT_MIN)

        pixels = image.reshape(-1, 3)
        keep = ~is_grass.reshape(-1)
        kept_pixels = pixels[keep]

        # If almost everything got excluded (box is nearly all pitch -- heavy occlusion,
        # a bad box, or a genuinely green/olive kit), fall back to the full crop rather
        # than return a median of a handful of pixels.
        if len(kept_pixels) < 0.15 * len(pixels):
            kept_pixels = pixels

        return np.median(kept_pixels, axis=0)

    def assign_team_colors_from_samples(self, player_colors_dict):
        """
        Lower-level entry point: takes a pre-computed {track_id: color} mapping (e.g.
        each player's color already aggregated/medianed across several frames by the
        caller) and does the actual team clustering + referee-outlier detection.
        Separated from assign_team_colors() so the pipeline can aggregate samples across
        multiple frames for stability instead of trusting a single frame.
        """
        track_ids = list(player_colors_dict.keys())
        player_colors = np.array([player_colors_dict[tid] for tid in track_ids])

        kmeans = KMeans(n_clusters=2, init="k-means++", n_init=10)
        kmeans.fit(player_colors)
        self.kmeans = kmeans

        self.team_colors[1] = kmeans.cluster_centers_[0]
        self.team_colors[2] = kmeans.cluster_centers_[1]

        # distance from each person's color to their nearest team-cluster center — a
        # referee's kit color won't match either team well, so this distance spikes for
        # them specifically. Threshold is data-driven (mean + 2*std), not a hardcoded guess.
        distances = []
        for color in player_colors:
            label = kmeans.predict(color.reshape(1, -1))[0]
            distances.append(np.linalg.norm(color - kmeans.cluster_centers_[label]))
        distances = np.array(distances)
        self.outlier_threshold = distances.mean() + 2 * distances.std() if len(distances) > 1 else np.inf

        for track_id, color in zip(track_ids, player_colors):
            label = kmeans.predict(color.reshape(1, -1))[0]
            dist = np.linalg.norm(color - kmeans.cluster_centers_[label])
            self.player_team_dict[track_id] = "referee" if dist > self.outlier_threshold else int(label) + 1

    def assign_team_colors(self, frame, player_detections):
        """Single-frame convenience wrapper around assign_team_colors_from_samples()."""
        player_colors = {
            track_id: self.get_player_color(frame, detection["bbox"])
            for track_id, detection in player_detections.items()
        }
        self.assign_team_colors_from_samples(player_colors)

    def get_player_team(self, frame, player_bbox, player_id):
        if player_id in self.player_team_dict:
            return self.player_team_dict[player_id]

        color = self.get_player_color(frame, player_bbox)
        label = self.kmeans.predict(color.reshape(1, -1))[0]
        dist = np.linalg.norm(color - self.kmeans.cluster_centers_[label])

        team = "referee" if (self.outlier_threshold is not None and dist > self.outlier_threshold) else int(label) + 1
        self.player_team_dict[player_id] = team
        return team


In [ ]:
%%writefile player_ball_assigner/player_ball_assigner.py
import sys

sys.path.append('/content')
from utils.bbox_utils import get_center_of_bbox


class PlayerBallAssigner:
    def __init__(self, max_player_ball_distance=70):
        self.max_player_ball_distance = max_player_ball_distance

    def assign_ball_to_player(self, players, ball_bbox):
        ball_x, ball_y = get_center_of_bbox(ball_bbox)
        minimum_distance = float("inf")
        assigned_player = -1

        for player_id, player in players.items():
            bbox = player["bbox"]
            distance_left = ((bbox[0] - ball_x) ** 2 + (bbox[-1] - ball_y) ** 2) ** 0.5
            distance_right = ((bbox[2] - ball_x) ** 2 + (bbox[-1] - ball_y) ** 2) ** 0.5
            distance = min(distance_left, distance_right)

            if distance < self.max_player_ball_distance and distance < minimum_distance:
                minimum_distance = distance
                assigned_player = player_id

        return assigned_player


In [ ]:
%%writefile camera_movement_estimator/camera_movement_estimator.py
import sys
sys.path.append('/content')

import cv2
import numpy as np

from utils.bbox_utils import measure_distance


class CameraMovementEstimator:
    """
    Estimates how much the CAMERA itself moved between consecutive frames (pan/tilt/
    drift), using sparse optical flow tracked on background features -- not players --
    so that "how far a player moved" can later be separated from "how far the camera
    moved and dragged everything in the frame along with it."

    On this project's footage the camera is a fixed, static overhead rig (the whole
    pitch stays framed identically across the whole clip), so in practice this should
    come out close to [0, 0] every frame. It's still worth running rather than assuming
    that: if a future clip comes from a genuinely panning or drone-drifting camera,
    speed/distance numbers would otherwise silently include the camera's own motion.
    """

    def __init__(self, first_frame):
        self.min_distance = 5

        # Only look for trackable features in narrow strips along the very top and
        # bottom of the frame. In a full-pitch overhead shot, players are rarely up
        # against those edges, so these strips are much more likely to be genuine
        # static background (stadium structure, advertising boards, empty grass)
        # whose only frame-to-frame motion is the camera's own.
        first_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
        h, w = first_gray.shape
        mask_features = np.zeros_like(first_gray)
        mask_features[0:int(h * 0.08), :] = 1
        mask_features[int(h * 0.92):h, :] = 1

        self.feature_params = dict(maxCorners=100, qualityLevel=0.3, minDistance=3, blockSize=7, mask=mask_features)
        self.lk_params = dict(
            winSize=(15, 15),
            maxLevel=2,
            criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03),
        )

    def get_camera_movement(self, frames):
        """Returns one [dx, dy] per frame: how far the camera moved since the PREVIOUS
        frame (pixels). Frame 0 is always [0, 0] (nothing to compare it to)."""
        camera_movement = [[0, 0] for _ in range(len(frames))]

        old_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
        old_features = cv2.goodFeaturesToTrack(old_gray, **self.feature_params)

        for frame_num in range(1, len(frames)):
            new_gray = cv2.cvtColor(frames[frame_num], cv2.COLOR_BGR2GRAY)

            if old_features is None or len(old_features) == 0:
                old_features = cv2.goodFeaturesToTrack(old_gray, **self.feature_params)
                if old_features is None:
                    old_gray = new_gray
                    continue

            new_features, status, _ = cv2.calcOpticalFlowPyrLK(old_gray, new_gray, old_features, None, **self.lk_params)

            if new_features is None or status is None:
                old_gray = new_gray
                old_features = cv2.goodFeaturesToTrack(old_gray, **self.feature_params)
                continue

            status = status.flatten()
            good_new = new_features[status == 1]
            good_old = old_features[status == 1]

            # take the single largest-displacement feature as the camera's movement
            # estimate, not an average -- an average gets pulled toward zero by
            # features that happen to sit on something that moved independently
            # (e.g. a player who wandered into the tracked strip), while the true
            # background motion is consistent and shows up as the dominant shift.
            max_distance = 0.0
            camera_movement_x, camera_movement_y = 0.0, 0.0
            for new, old in zip(good_new, good_old):
                new_pt, old_pt = new.ravel(), old.ravel()
                distance = measure_distance(new_pt, old_pt)
                if distance > max_distance:
                    max_distance = distance
                    camera_movement_x = old_pt[0] - new_pt[0]
                    camera_movement_y = old_pt[1] - new_pt[1]

            if max_distance > self.min_distance:
                camera_movement[frame_num] = [camera_movement_x, camera_movement_y]
                old_features = cv2.goodFeaturesToTrack(new_gray, **self.feature_params)
            else:
                old_features = good_new.reshape(-1, 1, 2) if len(good_new) > 0 else cv2.goodFeaturesToTrack(new_gray, **self.feature_params)

            old_gray = new_gray

        return camera_movement

    def adjust_positions_to_tracks(self, tracks, camera_movement):
        """
        Subtracts the camera's own movement (accumulated from frame 0) from every
        tracked player/ball bbox, writing the result as "adjusted_bbox". What's left
        is each object's position relative to the fixed pitch, not the camera --
        which is what view_transformer and speed_and_distance_estimator need.
        """
        cumulative_x, cumulative_y = 0.0, 0.0
        for frame_num in range(len(camera_movement)):
            cumulative_x += camera_movement[frame_num][0]
            cumulative_y += camera_movement[frame_num][1]

            for obj_type in ["players", "ball"]:
                for track_id, track in tracks[obj_type][frame_num].items():
                    bbox = track["bbox"]
                    tracks[obj_type][frame_num][track_id]["adjusted_bbox"] = [
                        bbox[0] - cumulative_x,
                        bbox[1] - cumulative_y,
                        bbox[2] - cumulative_x,
                        bbox[3] - cumulative_y,
                    ]


In [ ]:
%%writefile view_transformer/view_transformer.py
import sys
sys.path.append('/content')

import cv2
import numpy as np

from utils.bbox_utils import get_center_of_bbox, get_foot_position


class ViewTransformer:
    """
    Maps pixel positions from the video into real-world pitch coordinates (meters),
    via a homography computed once from a handful of known reference points -- e.g.
    the pitch's four corners -- whose pixel location you read off a frame, paired
    with their known real-world location on an actual pitch.

    This project's footage is a single, effectively static full-pitch overhead shot
    (the whole pitch is visible in every frame), so ONE homography, computed once,
    covers the entire clip -- unlike a panning/zooming broadcast camera, which would
    need this recomputed per frame or per shot.

    IMPORTANT: pixel_points must be given in order going around the pitch boundary
    (e.g. top-left, top-right, bottom-right, bottom-left) -- not paired diagonally --
    since they're also used to build the polygon that decides whether a given point
    is inside the calibrated pitch region at all.
    """

    def __init__(self, pixel_points, target_points):
        pixel_points = np.array(pixel_points, dtype=np.float32)
        target_points = np.array(target_points, dtype=np.float32)
        if len(pixel_points) < 4 or len(pixel_points) != len(target_points):
            raise ValueError("Need at least 4 matching pixel/target reference point pairs")

        self.pixel_polygon = pixel_points
        homography, _ = cv2.findHomography(pixel_points, target_points)
        if homography is None:
            raise ValueError("Could not compute a homography from the given reference points "
                              "-- check they aren't collinear or duplicated")
        self.perspective_transformer = homography

    def transform_point(self, point):
        """
        point: (x, y) pixel coordinate. Returns the corresponding (x, y) real-world
        pitch coordinate in meters, or None if the point falls outside the reference
        polygon. Extrapolating a homography far outside the region it was calibrated
        on gives meaningless (sometimes wildly wrong) results, so out-of-bounds points
        are refused rather than silently "transformed" into garbage.
        """
        p = (float(point[0]), float(point[1]))
        is_inside = cv2.pointPolygonTest(self.pixel_polygon, p, False) >= 0
        if not is_inside:
            return None

        reshaped = np.array([point], dtype=np.float32).reshape(-1, 1, 2)
        transformed = cv2.perspectiveTransform(reshaped, self.perspective_transformer)
        return transformed.reshape(-1, 2)[0]

    def transform_tracks(self, tracks):
        """
        Adds a "position_transformed" key (real-world [x, y] in meters, or None if
        outside the calibrated pitch region) to every player/ball track, using each
        object's camera-motion-adjusted position if camera_movement_estimator has
        already run (falls back to the raw bbox otherwise): foot position for players
        (where they're actually standing on the pitch), center for the ball.
        """
        for obj_type in ["players", "ball"]:
            for frame_num, frame_tracks in enumerate(tracks[obj_type]):
                for track_id, track in frame_tracks.items():
                    bbox = track.get("adjusted_bbox", track["bbox"])
                    position = get_foot_position(bbox) if obj_type == "players" else get_center_of_bbox(bbox)
                    transformed = self.transform_point(position)
                    tracks[obj_type][frame_num][track_id]["position_transformed"] = (
                        transformed.tolist() if transformed is not None else None
                    )


In [ ]:
%%writefile speed_and_distance_estimator/speed_and_distance_estimator.py
import sys
sys.path.append('/content')

import cv2

from utils.bbox_utils import measure_distance, get_foot_position


class SpeedAndDistanceEstimator:
    """
    Turns each player's real-world (meters) position over time into running distance
    covered (meters) and instantaneous speed (km/h). Computed over a rolling window of
    several frames rather than frame-to-frame -- frame-to-frame position jitter (a few
    pixels of detection noise, which becomes a few centimeters of real-world noise)
    would otherwise translate into wildly unstable speed readings.
    """

    def __init__(self, frame_window=5, fps=25):
        self.frame_window = frame_window
        self.fps = fps

    def add_speed_and_distance(self, tracks):
        total_distance = {}
        n_frames = len(tracks["players"])

        for frame_num in range(0, n_frames, self.frame_window):
            last_frame = min(frame_num + self.frame_window, n_frames - 1)
            if last_frame == frame_num:
                continue

            for track_id, track in tracks["players"][frame_num].items():
                if track_id not in tracks["players"][last_frame]:
                    continue

                start_pos = track.get("position_transformed")
                end_pos = tracks["players"][last_frame][track_id].get("position_transformed")
                if start_pos is None or end_pos is None:
                    continue

                distance_covered = measure_distance(start_pos, end_pos)
                time_elapsed = (last_frame - frame_num) / self.fps
                if time_elapsed <= 0:
                    continue

                speed_kmph = (distance_covered / time_elapsed) * 3.6
                total_distance[track_id] = total_distance.get(track_id, 0.0) + distance_covered

                # +1 so the window's own last frame gets written too -- otherwise the
                # very last frame of the whole clip (where last_frame == n_frames - 1
                # and there's no further window to start from it) would never get a
                # speed/distance value at all. Consecutive windows' ranges touch at
                # exactly one frame (this window's last_frame == the next window's
                # frame_num); that frame simply gets overwritten by the next window's
                # value a moment later, which is harmless.
                for fn in range(frame_num, last_frame + 1):
                    if track_id in tracks["players"][fn]:
                        tracks["players"][fn][track_id]["speed"] = speed_kmph
                        tracks["players"][fn][track_id]["distance"] = total_distance[track_id]

    def draw_speed_and_distance(self, frames, tracks):
        output = []
        for frame_num, frame in enumerate(frames):
            frame = frame.copy()
            for _, track in tracks["players"][frame_num].items():
                if "speed" not in track:
                    continue
                bbox = track.get("adjusted_bbox", track["bbox"])
                x, y = get_foot_position(bbox)
                y += 40
                cv2.putText(frame, f"{track['speed']:.1f} km/h", (int(x), int(y)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 2)
                y += 15
                cv2.putText(frame, f"{track['distance']:.1f} m", (int(x), int(y)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 2)
            output.append(frame)
        return output


In [ ]:
%%writefile formation_graph/formation_graph.py
import sys
sys.path.append('/content')

import cv2
import numpy as np


class FormationGraphDrawer:
    """
    Renders a flat, to-scale top-down "formation graph" -- the classic broadcast
    tactical-overlay look: each team's players as colored dots on a schematic pitch,
    connected to their nearest teammates by lines. Uses the REAL-WORLD (meters)
    positions from Stage 3's homography, not raw video pixels -- so the graph is a
    true-to-scale, undistorted view of the actual shape, independent of camera angle.

    Deliberately rendered as its own separate video rather than drawn on top of the
    drone footage: the broadcast graphics this is modeled on are shown as a separate
    schematic too, and it avoids the graph's lines/dots competing for the same pixels
    as the tracking/speed overlay already drawn on the main Stage 2/3 video.
    """

    TEAM_COLORS = {1: (60, 60, 220), 2: (220, 140, 40)}
    REFEREE_COLOR = (255, 255, 255)
    BALL_COLOR = (0, 255, 255)

    def __init__(self, pitch_length_m, pitch_width_m, scale=10, margin=40, n_nearest=2):
        self.pitch_length_m = pitch_length_m
        self.pitch_width_m = pitch_width_m
        self.scale = scale  # canvas pixels per real-world meter
        self.margin = margin
        self.n_nearest = n_nearest  # each player connects to their N nearest teammates

        self.canvas_w = int(pitch_length_m * scale) + 2 * margin
        self.canvas_h = int(pitch_width_m * scale) + 2 * margin
        self.pitch_template = self._draw_pitch_template()

    def _to_canvas(self, point):
        x, y = point
        return int(round(x * self.scale + self.margin)), int(round(y * self.scale + self.margin))

    def _draw_pitch_template(self):
        img = np.zeros((self.canvas_h, self.canvas_w, 3), dtype=np.uint8)
        img[:, :] = (60, 140, 60)

        tl = self._to_canvas((0, 0))
        br = self._to_canvas((self.pitch_length_m, self.pitch_width_m))
        cv2.rectangle(img, tl, br, (255, 255, 255), 2)

        halfway_top = self._to_canvas((self.pitch_length_m / 2, 0))
        halfway_bottom = self._to_canvas((self.pitch_length_m / 2, self.pitch_width_m))
        cv2.line(img, halfway_top, halfway_bottom, (255, 255, 255), 2)

        center = self._to_canvas((self.pitch_length_m / 2, self.pitch_width_m / 2))
        cv2.circle(img, center, int(9.15 * self.scale), (255, 255, 255), 2)
        cv2.circle(img, center, 3, (255, 255, 255), -1)

        return img

    def _nearest_teammate_edges(self, positions):
        """positions: {track_id: (x, y)} for ONE team. Returns a set of (id_a, id_b)
        pairs -- each player connected to their N nearest teammates, deduplicated
        regardless of which end found the pair first."""
        ids = list(positions.keys())
        edges = set()
        for a in ids:
            pa = np.array(positions[a])
            dists = sorted(
                (np.linalg.norm(pa - np.array(positions[b])), b) for b in ids if b != a
            )
            for _, b in dists[:self.n_nearest]:
                edges.add(tuple(sorted((a, b))))
        return edges

    def draw_frame(self, player_track, ball_track=None):
        frame = self.pitch_template.copy()

        positions_by_team = {1: {}, 2: {}}
        referee_positions = []

        for track_id, track in player_track.items():
            pos = track.get("position_transformed")
            if pos is None:
                continue
            team = track.get("team")
            if team in (1, 2):
                positions_by_team[team][track_id] = pos
            elif team == "referee":
                referee_positions.append(pos)

        for team, positions in positions_by_team.items():
            color = self.TEAM_COLORS[team]
            if len(positions) >= 2:
                for a, b in self._nearest_teammate_edges(positions):
                    cv2.line(frame, self._to_canvas(positions[a]), self._to_canvas(positions[b]),
                              color, 1, lineType=cv2.LINE_AA)
            for pos in positions.values():
                p = self._to_canvas(pos)
                cv2.circle(frame, p, 6, color, -1)
                cv2.circle(frame, p, 6, (0, 0, 0), 1)

        for pos in referee_positions:
            p = self._to_canvas(pos)
            cv2.circle(frame, p, 5, self.REFEREE_COLOR, -1)
            cv2.circle(frame, p, 5, (0, 0, 0), 1)

        if ball_track:
            for ball in ball_track.values():
                pos = ball.get("position_transformed")
                if pos is not None:
                    cv2.circle(frame, self._to_canvas(pos), 4, self.BALL_COLOR, -1)

        return frame

    def draw_all(self, tracks):
        output = []
        for frame_num, player_track in enumerate(tracks["players"]):
            ball_track = tracks["ball"][frame_num] if frame_num < len(tracks["ball"]) else None
            output.append(self.draw_frame(player_track, ball_track))
        return output


## Run detection, tracking, team assignment, possession

Same as Stage 2 — this part is already verified working, unchanged here.

In [ ]:
import sys
sys.path.append('/content')
import numpy as np
import pandas as pd
import cv2

from utils.video_utils import read_video, save_video, get_native_frame
from trackers.tracker import Tracker
from team_assigner.team_assigner import TeamAssigner
from player_ball_assigner.player_ball_assigner import PlayerBallAssigner
from camera_movement_estimator.camera_movement_estimator import CameraMovementEstimator
from view_transformer.view_transformer import ViewTransformer
from speed_and_distance_estimator.speed_and_distance_estimator import SpeedAndDistanceEstimator

video_frames = read_video(clip_path)
print(f"Loaded {len(video_frames)} frames")

tracker = Tracker(model_path)
tracks = tracker.get_object_tracks(video_frames)
print("Detection + tracking done")

In [ ]:
ball_positions = [x.get(1, {}).get("bbox", [np.nan] * 4) for x in tracks["ball"]]
df_ball = pd.DataFrame(ball_positions, columns=["x1", "y1", "x2", "y2"])
df_ball = df_ball.interpolate().bfill()
tracks["ball"] = [{1: {"bbox": row}} for row in df_ball.to_numpy().tolist()]
print("Ball positions interpolated")

In [ ]:
native_frame0 = get_native_frame(clip_path, 0)
scale = native_frame0.shape[1] / video_frames[0].shape[1]

def _scale_bbox(bbox, s):
    return [c * s for c in bbox]

team_assigner = TeamAssigner()

n_frames = len(tracks["players"])
sample_frame_nums = sorted(set(min(n_frames - 1, int(n_frames * f)) for f in [0.0, 0.2, 0.4, 0.6, 0.8]))

per_player_samples = {}
for fn in sample_frame_nums:
    native = get_native_frame(clip_path, fn)
    for player_id, track in tracks["players"][fn].items():
        color = team_assigner.get_player_color(native, _scale_bbox(track["bbox"], scale))
        per_player_samples.setdefault(player_id, []).append(color)

aggregate_colors = {tid: np.median(np.array(colors), axis=0) for tid, colors in per_player_samples.items()}
team_assigner.assign_team_colors_from_samples(aggregate_colors)

for frame_num, player_track in enumerate(tracks["players"]):
    native_frame = None
    for player_id, track in player_track.items():
        if player_id in team_assigner.player_team_dict:
            team = team_assigner.player_team_dict[player_id]
        else:
            if native_frame is None:
                native_frame = get_native_frame(clip_path, frame_num)
            team = team_assigner.get_player_team(native_frame, _scale_bbox(track["bbox"], scale), player_id)

        tracks["players"][frame_num][player_id]["team"] = team
        if team == "referee":
            tracks["players"][frame_num][player_id]["team_color"] = (255, 255, 255)
        else:
            tracks["players"][frame_num][player_id]["team_color"] = tuple(
                int(c) for c in team_assigner.team_colors[team]
            )

print("Team colors (BGR):", team_assigner.team_colors)
referee_count = sum(1 for p in tracks["players"][0].values() if p.get("team") == "referee")
print(f"Frame 0: {len(tracks['players'][0])} people detected, {referee_count} flagged as referee")

In [ ]:
ball_assigner = PlayerBallAssigner()
team_ball_control = []
for frame_num, player_track in enumerate(tracks["players"]):
    ball_bbox = tracks["ball"][frame_num][1]["bbox"]
    assigned_player = ball_assigner.assign_ball_to_player(player_track, ball_bbox)

    if assigned_player != -1:
        tracks["players"][frame_num][assigned_player]["has_ball"] = True
        assigned_team = tracks["players"][frame_num][assigned_player].get("team")
        if assigned_team in (1, 2):
            team_ball_control.append(assigned_team)
        else:
            team_ball_control.append(team_ball_control[-1] if team_ball_control else 0)
    else:
        team_ball_control.append(team_ball_control[-1] if team_ball_control else 0)

team_ball_control = np.array(team_ball_control)
print("Possession computed")

## New in Stage 3: camera motion compensation

Estimates how much the camera itself moved frame-to-frame (should be ~0 for this clip's static shot) and adjusts every tracked position to cancel that out, so what's left is each object's own motion relative to the pitch.

In [ ]:
camera_estimator = CameraMovementEstimator(video_frames[0])
camera_movement = camera_estimator.get_camera_movement(video_frames)
camera_estimator.adjust_positions_to_tracks(tracks, camera_movement)

max_movement = max(abs(dx) + abs(dy) for dx, dy in camera_movement)
print(f"Max per-frame camera movement detected: {max_movement:.1f}px "
      f"({'looks static, as expected' if max_movement < 5 else 'camera is actually moving in this clip'})")

## New in Stage 3: calibrate the pixel-to-pitch homography

This is the one step that needs a human eye. The cell below saves and displays the first frame with a pixel coordinate grid overlaid. Look at it and note the approximate **pixel (x, y)** of the pitch's four corners (where each touchline meets each byline), going around in order — e.g. top-left, top-right, bottom-right, bottom-left. Don't skip around (top-left → bottom-right → top-right would break the calibration).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, ax = plt.subplots(figsize=(16, 9))
ax.imshow(cv2.cvtColor(native_frame0, cv2.COLOR_BGR2RGB))
ax.xaxis.set_major_locator(ticker.MultipleLocator(200))
ax.yaxis.set_major_locator(ticker.MultipleLocator(200))
ax.grid(True, color='yellow', alpha=0.5, linewidth=0.5)
ax.set_title(f"Native frame 0 ({native_frame0.shape[1]}x{native_frame0.shape[0]}px) -- read off the 4 pitch corners' pixel coordinates")
plt.savefig('frame0_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Frame size: {native_frame0.shape[1]} wide x {native_frame0.shape[0]} tall")

In [ ]:
# Read off the grid image above (native 3840x2160 frame). You reported the corners as
# (420,175), (450,2000), (3400,2050), (3400,100) -- that's top-left, bottom-left,
# bottom-right, top-right in image terms. Reordered below to top-left, top-right,
# bottom-right, bottom-left to match REAL_CORNERS' order -- same 4 points, just walked
# around the boundary in the other direction so the two lists line up correctly.
PIXEL_CORNERS = [
    [420, 175],    # top-left corner of the pitch, in pixels
    [3400, 100],   # top-right
    [3400, 2050],  # bottom-right
    [450, 2000],   # bottom-left
]

# Real pitch dimensions in meters -- FIFA standard is 105 x 68. Change these if you
# know this pitch's actual dimensions.
PITCH_LENGTH_M = 105
PITCH_WIDTH_M = 68

REAL_CORNERS = [
    [0, 0],
    [PITCH_LENGTH_M, 0],
    [PITCH_LENGTH_M, PITCH_WIDTH_M],
    [0, PITCH_WIDTH_M],
]

view_transformer = ViewTransformer(PIXEL_CORNERS, REAL_CORNERS)
view_transformer.transform_tracks(tracks)

in_bounds = sum(
    1 for p in tracks["players"][0].values() if p.get("position_transformed") is not None
)
print(f"Frame 0: {in_bounds}/{len(tracks['players'][0])} players landed inside the calibrated pitch region")
print("If that number looks low, your PIXEL_CORNERS likely need adjusting -- re-check the")
print("grid image above and make sure the order goes around the boundary, not diagonally.")

## New in Stage 3: speed and distance

Uses the real-world (meters) positions from the homography above to compute each player's running distance covered and current speed, in a rolling window (not frame-to-frame, which would be far too noisy).

In [ ]:
speed_estimator = SpeedAndDistanceEstimator(frame_window=5, fps=25)
speed_estimator.add_speed_and_distance(tracks)
print("Speed and distance computed")

In [ ]:
output_frames = tracker.draw_annotations(video_frames, tracks, team_ball_control)
output_frames = speed_estimator.draw_speed_and_distance(output_frames, tracks)
save_video(output_frames, 'stage3_output.mp4', fps=25)
print("Saved stage3_output.mp4")
files.download('stage3_output.mp4')

## New in Stage 4: formation / network graph

A flat, to-scale top-down diagram: each team's players as colored dots (connected to their nearest 2 teammates by lines), plus the ball, using the real-world (meters) positions from the homography above. Rendered as its own video, separate from the main tracking video.

In [ ]:
from formation_graph.formation_graph import FormationGraphDrawer

formation_drawer = FormationGraphDrawer(pitch_length_m=PITCH_LENGTH_M, pitch_width_m=PITCH_WIDTH_M,
                                         scale=10, margin=40, n_nearest=2)
formation_frames = formation_drawer.draw_all(tracks)
save_video(formation_frames, 'stage4_formation_graph.mp4', fps=25)
print("Saved stage4_formation_graph.mp4")
files.download('stage4_formation_graph.mp4')

## Bring the code home

Download the actual pipeline files to drop into your local project folder.

In [ ]:
import zipfile, os

with zipfile.ZipFile('stage4_modules.zip', 'w') as z:
    for folder in ['utils', 'trackers', 'team_assigner', 'player_ball_assigner',
                    'camera_movement_estimator', 'view_transformer', 'speed_and_distance_estimator',
                    'formation_graph']:
        for root, _, filenames in os.walk(folder):
            for fn in filenames:
                z.write(os.path.join(root, fn))
files.download('stage4_modules.zip')
print("Downloaded stage4_modules.zip — unzip into your local project folder root.")

### Done for now

You should have downloaded `stage3_output.mp4` (tracking + speed/distance), `stage4_formation_graph.mp4` (the new top-down network diagram), and `stage4_modules.zip`.

Watch `stage4_formation_graph.mp4` side by side with `stage3_output.mp4`: the dots should roughly mirror where players are in the tracking video (same relative layout), and each team's 2-nearest-teammate lines should look like a plausible, connected shape (not one giant tangle or totally disconnected dots) as players move.

Sanity-check the speed numbers too: human sprint speeds top out around 30-36 km/h, and a player rarely covers much more than ~10-13 km over a FULL 90-minute match — for this ~12 second clip, distance per player should be a small fraction of that. If speeds look wildly too high or too low, the most likely cause is `PIXEL_CORNERS` — recheck them against the grid image.

**Known limitation, not fixed here:** goalkeepers usually wear a different kit color than their own outfield teammates, so team-color clustering will likely misclassify them (flag them as the "referee" outlier, or assign them to the wrong team) — which also means they'll likely show up in the formation graph as a white dot with no team edges, rather than correctly anchored in their own team's shape. Not solved automatically in this version — a manual override per match is the likely path rather than a general fix.

**On generalizing to other clips:** `PIXEL_CORNERS` is specific to this clip's exact camera framing — a different clip needs its own 4 corners read off its own grid image, by design. The team-color grass-hue exclusion in `team_assigner.py` was tuned against this footage's grass tone, and different lighting/turf could need those thresholds adjusted — worth checking on the next clip, not assuming it'll just work.